## Transistors --> Primitive Logic (NAND, NOR, NOT, AND, OR, XOR)

- The purpose of this section is to take us from Transistors --> Nand gates
    - Nand gates are universal building blocks in computing
    - We will not go into the physics of creating transistors, or how they are doped
    - As far as we are concerned, we treat transistors as the most atomic building block
    - From this basic hardware atom, springs the entire universe of computing

- So what exactly is a transistor?
    - Think of transistors as switches that responds in a fixed way to voltage
    - You can think of these as basic mechanical switches with 2 inputs; (i) One input comes from the main power source, which is always 1 when the power is on, and (ii) the second input is what drives behaviour, and it is supplied by the output wire of another transistor, an external input pin (like a keyboard switch), or the system clock.

- A transistor can output 2 states:
    - Either it passes through the power that is being supplied, which reads as 1
    - Or it passes GROUND, which reads as None
    - Let's fix the 2 states here as global constants

### Basic Transistors: NMOS + PMOS

In [ ]:
- There are 2 basic types of transistors we need to know about, that form the building blocks for basic logical gates
    - A PMOS transistor 
        - P-type Metal-Oxide-Semiconductor
        - Also known as a "Pull-Up Team"
        
    - A NMOS transistor
        - N-type Metal-Oxide-Semiconductor
        - Also known as a "Pull-Down Team"

In [ ]:
from utils import *

def pmos(gate_signal: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    """
    Pass through source signal when gate_signal is False (i.e. low power)
    None when gate_signal is True (i.e. high power)
    """
    return source if not gate_signal else None

def nmos(gate_signal: TRANSISTOR_OUTPUT, source: bool = POWER) -> POWER | GROUND:
    """
    Pass through source signal when gate_signal is True (i.e. high power)
    None when gate_signal is False (i.e low power)
    """
    return source if gate_signal else None

In [ ]:
print(
    pmos(True), pmos(False),
    nmos(True), nmos(False)
)

### PMOS and NMOS to Logic

- From these 2 signals, we can start building out logic gates!

- Fun fact; in electronics, NAND and NOR are much simpler to build than NOT/AND/OR 
    - So we will start by building out NAND and NOR using both PMOS and NMOS transistors

- In the code below, we replicate the effect of the transistor -> logic gates via code. 
    - One source of confusion: you might notice we use logical expressions `and`, `or` to build out these logic gates
    - It probably seems circular that we are creating logic gates from logic expressions, but keep in mind that this is just a representation of how the transistor works. 
    - In the physical world, we rely on physical arrangement of the circuit (parallel, series) to derive this effect

- Every function below comes with an ASCII sketch of the circuit to show what I mean

In [ ]:
def pmos_NAND(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                     POWER (1)
                  ┌─────┴─────┐
                  │           │
     Input1 --> [PMOS A]    [PMOS B] <-- Input2
                  │           │
                  └─────┬─────┘
                        ├─── OUTPUT (Out)
                        │
                    [Resistor]
                        │
                    GROUND (0)
    '''
    # If input1 and input2 are both True, both PMOS return None, and so no output is returned
    # Otherwise, so long as one of input1/input2 are False, the power flows through one of the PMOS paths and 
    # output returns True
    pmos_A = pmos(input1, source)
    pmos_B = pmos(input2, source)
    
    return POWER if pmos_A or pmos_B else GROUND

def pmos_NOR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
        POWER (1)
            │
        [PMOS A] <-- Input1
            │             
        [PMOS B] <-- Input2
            │
            ├─── OUTPUT (Out)
            │
        [Resistor]
            │
        GROUND (0)
    '''
    # If both input1 and input2 are None, both PMOS transistors let power through, and Output is POWER
    # Else if either input1 or input2 are True, the circuit breaks, and ground is returned
    pmosA = pmos(input1, source)
    pmosB = pmos(input2, pmosA)

    return POWER if pmos_B else GROUND

def pmos_NOT(input1: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
        POWER (1)
            │
         [PMOS] <-- Input1
            │
            ├─── OUTPUT (Out)
            │
        [Resistor]
            │
        GROUND (0)
    '''
    ## PMOS negates the input by construction, because it outputs 0 if input is 1
    out1 = pmos(input1, source)
    return POWER if out1 else GROUND

def pmos_AND(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                     POWER (1)
                  ┌─────┴─────┐
                  │           │
     Input1 --> [PMOS A]    [PMOS B] <-- Input2
                  │           │
                  └─────┬─────┘
                        ├─── NAND Output
                        │
                    [Resistor]
                        │
                    GROUND (0)
                        │
                     POWER (1)
                        │
                    [PMOS NOT] <-- NAND Output
                        │
                        ├─── OUTPUT (AND Out)
                        │
                    [Resistor]
                        │
                    GROUND (0)
    '''

    ## - If both input1 and input2 are 1, then the NAND Output is 0
    ##    - If the NAND output is 0, the NOT output is 1
    ## - Else the NAND output is 1
    ##    - If the NAND output is 1, the NOT output is 0
    ## - Therefore, the truth table becomes equivalent to AND
    ##    - (1,1) -> 1
    ##    - (0,1) -> 0
    ##    - (1,0) -> 0
    ##    - (0,0) -> 0

    nand_out = pmos_NAND(input1, input2, source=source)
    return pmos_NOT(nand_out, source=source)

def pmos_OR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                    POWER (1)
                        │
     Input1 -------> [PMOS A]
                        │             
     Input2 -------> [PMOS B]
                        │
                        ├─── Intermediate (NOR)
                        │
                    [Resistor]
                        │
                    GROUND (0)
                        │
                     POWER (1)
                        │
                    [PMOS NOT] <-- Intermediate (NOR)
                        │
                        ├─── OUTPUT (OR Out)
                        │
                    [Resistor]
                        │
                    GROUND (0)
    '''
    ## - If both input1 and input2 are 0, then the NOR Output is 1
    ##    - If the NOR output is 1, the NOT output is 0
    ## - Else the NOR output is 0
    ##    - If the NOR output is 0, the NOT output is 1
    ## - Therefore, the truth table becomes equivalent to OR
    ##    - (1,1) -> 1
    ##    - (0,1) -> 1
    ##    - (1,0) -> 1
    ##    - (0,0) -> 0
    nor_out = pmos_NOR(input1, input2, source=source)
    return pmos_NOT(nor_out, source=source)

def pmos_XNOR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
        POWER (1)
            │
      ┌─────┴─────────────────────┐
      │ Branch 1 (0, 0)           │ Branch 2 (1, 1)
  [PMOS A]  <-- Input1        [PMOS A'] <-- NOT(Input1)
      │                           │
  [PMOS B]  <-- Input2        [PMOS B'] <-- NOT(Input2)
      └─────┬─────────────────────┘
            ├─── OUTPUT (Out)
            │
        [Resistor]
            │
        GROUND (0)
    '''
    ## - If (0, 0) --> 1
    ##    - PMOS A and B both return 1, PMOS A' and B' both return 0
    ##    - POWER hits output, returns 1
    ## - If (0, 1) --> 0
    ##    - A returns 1 and B returns 0, A' returns 0 and B' returns 1
    ##    - Both circuts break, returns 0
    ## - If (1, 0) --> 0
    ##    - Same as (0, 1) case
    ## - If (1, 1)
    ##    - A and B both return 0, A' and B' both return 1
    ##    - POWER hits output, returns 1

    input1_inv = pmos_NOT(input1, source)
    input2_inv = pmos_NOT(input2, source)

    pmos_A = pmos(input1, source)
    pmos_B = pmos(input2, pmos_A)

    pmos_A2 = pmos(input1_inv, source)
    pmos_B2 = pmos(input2_inv, pmos_A2)

    if b1_out is not None:
        return b1_out
    elif b2_out is not None:
        return b2_out
    else:
        return GROUND

def pmos_XOR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, source: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
              Input1    Input2
                 │         │
        ┌────────┼─────────┼────────┐
        │        │         │        │
        ▼        ▼         ▼        ▼
     ┌──────────────┐   ┌──────────────┐
     │   pmos_OR    │   │  pmos_NAND   │
     └──────┬───────┘   └──────┬───────┘
            │                  │
         OR Out             NAND Out
            │                  │
            └────────┬─────────┘
                     │
                     ▼
             ┌──────────────┐
             │   pmos_AND   │
             └──────┬───────┘
                    │
                    ▼
                 OUTPUT
    '''
    ## - If (0, 0) --> 0
    ##    - PMOS OR returns 0, PMOS NAND returns 1
    ##    - PMOS AND returns 0
    ## - If (0, 1) --> 1
    ##    - PMOS OR returns 1, PMOS NAND returns 1
    ##    - PMOS AND returns 1    
    ## - If (1, 0) --> 1
    ##    - PMOS OR returns 1, PMOS NAND returns 1
    ##    - PMOS AND returns 1
    ## - If (1, 1) --> 0
    ##    - PMOS OR returns 1, PMOS NAND returns 0
    ##    - PMOS AND returns 0
    or_out = pmos_OR(input1, input2, source=source)
    nand_out = pmos_NAND(input1, input2, source=source)
    return pmos_AND(or_out, nand_out, source=source)

In [ ]:
def nmos_NAND(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
        POWER (1)
            │
        [Resistor]
            │
            ├─── OUTPUT (Out)
            │
        [NMOS A] <-- Input1
            │
        [NMOS B] <-- Input2
            │
        GROUND (0)
    '''
    ## If both input1 and input2 are 1, NMOS outputs 1. 
    ## Therefore, series goes to ground. Else output 1
    nmos_A = nmos(input1, GROUND)
    nmos_B = nmos(input2, nmos_A)
    return GROUND if nmos_B else power

def nmos_NOR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                    POWER (1)
                        │
                    [Resistor]
                        │
                        ├─── OUTPUT (Out)
                  ┌─────┴─────┐
                  │           │
     Input1 --> [NMOS A]    [NMOS B] <-- Input2
                  │           │
                  └─────┬─────┘
                        │
                    GROUND (0)
    '''
    ## If both input1 and input2 are 0, NMOS outputs 0. 
    ## Therefore, outputs 1. Else output GROUND
    nmos_A = nmos(input1, GROUND)
    nmos_B = nmos(input2, GROUND)
    return GROUND if (nmos_A or nmos_B) else power

def nmos_NOT(input1: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
        POWER (1)
            │
        [Resistor]
            │
            ├─── OUTPUT (Out)
            │
        [NMOS] <-- Input
            │
        GROUND (0)
    '''
    # When input1 is 0, NMOS outputs 0, which breaks the circuit to ground. Therefore, output is 1
    nmos_out = nmos(input1, GROUND)
    return GROUND if nmos_out else power

def nmos_AND(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
        POWER (1)
            │
        [Resistor]
            │
            ├─── Intermediate (NAND)
            │
        [NMOS A] <-- Input1
            │
        [NMOS B] <-- Input2
            │
        GROUND (0)
            
         POWER (1)
            │
        [Resistor]
            │
            ├─── OUTPUT (AND Out)
            │
        [NMOS C] <-- Intermediate (NAND)
            │
        GROUND (0)
    '''
    ## - If both input1 and input2 are 1, then the NAND Output is 0
    ##    - If the NAND output is 0, NMOS C output is 0, which lets the circuit output POWER
    ## - Else the NAND output is 1
    ##    - If the NAND output is 1, NMOC C output is 1, and the output goes to GROUND
    ## - Therefore, the truth table becomes equivalent to AND
    ##    - (1,1) -> 1
    ##    - (0,1) -> 0
    ##    - (1,0) -> 0
    ##    - (0,0) -> 0
    
    nand_out = nmos_NAND(input1, input2, power=power)
    return nmos_NOT(nand_out, power=power)
    return GROUND if not_out else POWER

def nmos_OR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                    POWER (1)
                        │
                    [Resistor]
                        │
                        ├─── Intermediate (NOR)
                  ┌─────┴─────┐
                  │           │
     Input1 --> [NMOS A]    [NMOS B] <-- Input2
                  │           │
                  └─────┬─────┘
                        │
                    GROUND (0)
                        
                     POWER (1)
                        │
                    [Resistor]
                        │
                        ├─── OUTPUT (OR Out)
                        │
                    [NMOS C] <-- Intermediate
                        │
                    GROUND (0)
    '''
    ## - If both input1 and input2 are 0, then the NOR Output is 1
    ##    - If the NOR output is 1, NMOS C output is 1, which pull circuit to GROUND
    ## - Else the NOR output is 0
    ##    - If the NOR output is 0, NMOS C output is 0, and circuit outputs POWER
    ## - Therefore, the truth table becomes equivalent to OR
    ##    - (1,1) -> 1
    ##    - (0,1) -> 1
    ##    - (1,0) -> 1
    ##    - (0,0) -> 0
    
    nor_out = nmos_NOR(input1, input2, power=power)
    return nmos_NOT(nor_out, power=power)

def nmos_XNOR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
        POWER (1)
            │
        [Resistor]
            │
            ├─── OUTPUT (Out)
            │
      ┌─────┴─────────────────────┐
      │ Branch 1                  │ Branch 2 
  [NMOS A'] <-- NOT(Input1)   [NMOS A]  <-- Input1
      │                           │
  [NMOS B]  <-- Input2        [NMOS B'] <-- NOT(Input2)
      └─────┬─────────────────────┘
            │
        GROUND (0)
    '''
    ## - If (0, 0) --> 1
    ##    - A returns 0, B' returns 1
    ##    - A' returns 1, B returns 0
    ##    - Both drain paths blocked, so output is 1
    ## - If (0, 1) --> 0
    ##    - A returns 0, B' returns 0
    ##    - A' returns 1, B returns 1
    ##    - Left drain path connected, so output is 0
    ## - If (1, 0) --> 0
    ##    - A returns 1, B' returns 1
    ##    - A' returns 0, B returns 0
    ##    - Right drain path connected, so output is 0
    ## - If (1, 1)
    ##    - A returns 1, B' returns 0 
    ##    - A' returns 0, B returns 1
    ##    - Both drain paths blocked, so output is 1
    
    input1_inv = nmos_NOT(input1)
    input2_inv = nmos_NOT(input2)

    nmos_A = nmos(input1, power)
    nmos_B2 = nmos(input2_inv, nmos_A)

    nmos_A2 = nmos(input1_inv, power)
    nmos_B = nmos(input2, nmos_A2)
    
    if nmos_B is not None or nmos_B2 is not None:
        return GROUND
    else:
        return POWER

def nmos_XOR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, ground: bool = GROUND) -> TRANSISTOR_OUTPUT:
    '''
    NMOS XOR constructed from fundamental NMOS building blocks:
    XOR = (A OR B) AND (A NAND B)
    
               Input1    Input2
                 │         │
        ┌────────┼─────────┼────────┐
        │        │         │        │
        ▼        ▼         ▼        ▼
     ┌──────────────┐   ┌──────────────┐
     │   nmos_OR    │   │  nmos_NAND   │
     └──────┬───────┘   └──────┬───────┘
            │                  │
         OR Out             NAND Out
            │                  │
            └────────┬─────────┘
                     │
                     ▼
             ┌──────────────┐
             │   nmos_AND   │
             └──────┬───────┘
                    │
                    ▼
                 OUTPUT
    '''
    ## - If (0, 0) --> 0
    ##    - NMOS OR returns 0, NMOS NAND returns 1
    ##    - NMOS AND returns 0
    ## - If (0, 1) --> 1
    ##    - NMOS OR returns 1, NMOS NAND returns 1
    ##    - NMOS AND returns 1    
    ## - If (1, 0) --> 1
    ##    - NMOS OR returns 1, NMOS NAND returns 1
    ##    - NMOS AND returns 1
    ## - If (1, 1) --> 0
    ##    - NMOS OR returns 1, NMOS NAND returns 0
    ##    - NMOS AND returns 0
    or_out = nmos_OR(input1, input2, ground=ground)
    nand_out = nmos_NAND(input1, input2, ground=ground)
    return nmos_AND(or_out, nand_out, ground=ground)

### CMOS

- We just showed how basic logical operations can be created from purely NMOS or PMOS transistors! But most modern computers don't use these. Instead, they use CMOS transistors!

- Why? 

- PMOS/NMOS transistors suffer from 2 big problems (i) Wasteful power drain and (ii) Asymmetric Signal Quality. These apply for all logic gates, but lets use NAND in PMOS and NMOS as a motivating example

- Wasteful power drain 
    - In PMOS NAND:
        - As long as either PMOS A or PMOS B receives an input of 0, it will let SOURCE pass through power
        - This relies entirely on the resistor between SOURCE and GROUND to block current going to ground
        - No matter how much resistance you have, $P = \frac{V^2}{R}$, so some power is always dissipated at resistor as heat
        - Put another way, in 3/4 of possible states `(0,0), (0,1), (1,0)`, this dissipation will cause heat dissipation, which creates an upper limit on the number of transistors you can put
    - In NMOS NAND:
        - In the NMOS case, when receiving a `(1, 1)`, the signal discharges straight to ground
        - So to output a 0, you need to drain electricity constantly into ground, which is exceedingly wasteful, and also generates a ton of heat

- Asymmetric Signal Quality
    - In PMOS NAND:
        - When receiving (1,1) input, both PMOS transistors open and break the circuit. 
        - HOWEVER, nothing ACTIVELY discharges the transistor. We rely entirely on charge draining through the resistor into GROUND
        - This means that while PMOS NAND signals 1 reliably, signalling 0 can be a bit sketchy because it is affected by drain rate. This is especially problematic when the clock speed is high and you need to charge and discharge quickly
    - In NMOS NAND:
        - When receiving (1,1) input, both NMOS transistors close the circuit to ground with 0 resistance
        - HOWEVER, for all other states, the NMOS transistors break the circuit, but the charging of the transistor back up to a 1 state takes a long time, because it goes through a resistor
        - This means that NMOS NAND signals 0 reliably, but signalling 1 can take time. Again, this affects how fast your computer clock speed can be!

- Therefore, to get the best of both worlds, it is ideal to combine both PMOS and NMOS into CMOS transistors!
    - CMOS isn't its own type of transistor per-se. It is simply a transistor combining the behaviours of PMOS and NMOS!

- We'll implement the same logic 5 gates using CMOS

In [ ]:
def cmos_NOT(input1: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                        POWER (1)
                            │
        Input1 ───────── [PMOS] (Pull-Up)
                            │
                            ├─── OUTPUT (Out)
                            │
        Input1 ───────── [NMOS] (Pull-Down)
                            │
                        GROUND (0)
    '''
    ## - If input1 is 0 (GROUND):
    ##    - PMOS turns ON, connecting OUTPUT directly to POWER
    ##    - NMOS turns OFF, blocking path to GROUND
    ##    - OUTPUT = POWER (1)
    ## - If input1 is 1 (POWER):
    ##    - PMOS turns OFF, blocking path to POWER
    ##    - NMOS turns ON, connecting OUTPUT directly to GROUND
    ##    - OUTPUT = GROUND (0)
    pmos_out = pmos(input1, power)
    nmos_out = nmos(input1, GROUND)

    if pmos_out is not None:
        return power
    if nmos_out is not None:
        return GROUND
    return None

def cmos_NAND(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                        POWER (1)
                     ┌──────┴──────┐
                     │             │
        Input1 ─── [PMOS A]     [PMOS B] ─── Input2
                     │             │
                     └──────┬──────┘
                            ├─── OUTPUT (Out)
                            │
        Input1 ───────── [NMOS A]
                            │
        Input2 ───────── [NMOS B]
                            │
                        GROUND (0)
    '''
    ## - PMOS transistors in parallel (Pull-Up Network)
    ## - NMOS transistors in series (Pull-Down Network)
    ## - If BOTH inputs are 1:
    ##    - Both PMOS turn OFF, both NMOS turn ON
    ##    - Path to POWER blocked; path to GROUND connected
    ##    - OUTPUT = GROUND (0)
    ## - Else with at least one 0:
    ##    - At least one PMOS turns ON, NMOS series path broken
    ##    - OUTPUT = POWER (1)
    pmos_A = pmos(input1, power)
    pmos_B = pmos(input2, power)

    nmos_A = nmos(input1, GROUND)
    nmos_B = nmos(input2, nmos_A)

    if pmos_A or pmos_B:
        return power
    if nmos_B is not None:
        return GROUND
    return None

def cmos_NOR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                        POWER (1)
                            │
        Input1 ───────── [PMOS A]
                            │
        Input2 ───────── [PMOS B]
                            │
                            ├─── OUTPUT (Out)
                     ┌──────┴──────┐
                     │             │
        Input1 ─── [NMOS A]     [NMOS B] ─── Input2
                     │             │
                     └──────┬──────┘
                            │
                        GROUND (0)
    '''
    ## - PMOS transistors in series (Pull-Up Network)
    ## - NMOS transistors in parallel (Pull-Down Network)
    ## - If BOTH inputs are 0:
    ##    - Both PMOS turn ON, both NMOS turn OFF
    ##    - Path to POWER connected; path to GROUND blocked
    ##    - OUTPUT = POWER (1)
    ## - Otherwise (at least one input is 1):
    ##    - At least one NMOS turns ON, PMOS series path broken
    ##    - OUTPUT = GROUND (0)
    pmos_A = pmos(input1, power)
    pmos_B = pmos(input2, pmos_A)

    nmos_A = nmos(input1, GROUND)
    nmos_B = nmos(input2, GROUND)

    if pmos_B is not None:
        return power
    if nmos_A or nmos_B:
        return GROUND
    return None

def cmos_AND(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                        POWER (1)
                     ┌──────┴──────┐
                     │             │
        Input1 ─── [PMOS A]     [PMOS B] ─── Input2
                     │             │
                     └──────┬──────┘
                            ├─── NAND Output
                            │
        Input1 ───────── [NMOS A]
                            │
        Input2 ───────── [NMOS B]
                            │
                        GROUND (0)
                            
                        POWER (1)
                            │
        NAND Out ─────── [PMOS C]
                            │
                            ├─── OUTPUT (AND Out)
                            │
        NAND Out ─────── [NMOS C]
                            │
                        GROUND (0)
    '''
    ## - If both inputs are 1:
    ##    -  NAND output is 0
    ##       - Then PMOS C is 1 and NMOS C is 0
    ##          - So OUTPUT (1)
    ## - Else:
    ##    -  NAND output is 1
    ##       - Then PMOS C is 0 and NMOS C is 1
    ##          - So GROUND (0)
    ## - Overall
    ##    - (1, 1) -> 1
    ##    - (1, 0) -> 0
    ##    - (0, 1) -> 0
    ##    - (0, 0) -> 0
    nand_out = cmos_NAND(input1, input2, power=power)
    return cmos_NOT(nand_out, power=power)

def cmos_OR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
                        POWER (1)
                            │
        Input1 ───────── [PMOS A]
                            │
        Input2 ───────── [PMOS B]
                            │
                            ├─── NOR Output
                     ┌──────┴──────┐
                     │             │
        Input1 ─── [NMOS A]     [NMOS B] ─── Input2
                     │             │
                     └──────┬──────┘
                            │
                        GROUND (0)
                            
                        POWER (1)
                            │
        NOR Out ──────── [PMOS C]
                            │
                            ├─── OUTPUT (OR Out)
                            │
        NOR Out ──────── [NMOS C]
                            │
                        GROUND (0)
    '''
    ## - If both inputs are 0:
    ##    -  NOR output is 1
    ##       - Then PMOS C is 0 and NMOS C is 1
    ##          - So GROUND (0)
    ## - Else:
    ##    -  NOR output is 0
    ##       - Then PMOS C is 1 and NMOS C is 0
    ##          - So POWER (1)
    ## - Overall
    ##    - (1, 1) -> 1
    ##    - (1, 0) -> 1
    ##    - (0, 1) -> 1
    ##    - (0, 0) -> 0
    nor_out = cmos_NOR(input1, input2, power=power)
    return cmos_NOT(nor_out, power=power)

def cmos_XNOR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, power: bool = POWER) -> TRANSISTOR_OUTPUT:
    '''
               POWER (1)
                   │
         ┌─────────┴─────────┐
         │ PMOS              │ PMOS 
     [PMOS A] Input1     [PMOS A'] <-- NOT(Input1)
         │                   │
     [PMOS B] Input2     [PMOS B'] <-- NOT(Input2)
         └─────────┬─────────┘
                   ├─── OUTPUT (Out)
         ┌─────────┴─────────┐
         │ NMOS              │ NMOS 
     [NMOS A'] NOT(Input1) [NMOS A]  <-- Input1
         │                   │
     [NMOS B] Input2       [NMOS B'] <-- NOT(Input2)
         └─────────┬─────────┘
                   │
               GROUND (0)
    '''
    ## - If (0, 0) --> 1
    ##    - PA returns 1, PB returns 1, PA2 returns 0, PB2 returns 0
    ##    - ✔ Power flows down left branch
    ##    - NA returns 0, NB2 returns 1, NA2 returns 1, NB returns 0
    ##    - ✘ Both drain branches are not connected 
    ##    - Output: POWER (1)
    ## - If (0, 1) --> 0
    ##    - PA returns 1, PB returns 0, PA2 returns 0, PB2 returns 1
    ##    - ✘ No powers to both paths 
    ##    - NA returns 0, NB2 returns 0, NA2 returns 1, NB returns 1
    ##    - ✔ Right drain branch connected
    ##    - Output: GROUND (0)
    ## - If (1, 0) --> 0
    ##    - PA returns 0, PB returns 1, PA2 returns 1, PB2 returns 0
    ##    - ✘ No powers to both paths 
    ##    - NA returns 1, NB2 returns 1, NA2 returns 0, NB returns 0
    ##    - ✔ Left drain branch connected
    ##    - Output: GROUND (0)
    ## - If (1, 1)
    ##    - PA returns 0, PB returns 0, PA2 returns 1, PB2 returns 1
    ##    - ✔ Power flows down right branch
    ##    - NA returns 1, NB2 returns 0, NA2 returns 0, NB returns 1
    ##    - ✘ Both drain branches are not connected 
    ##    - Output: POWER (1)

    input1_inv = cmos_NOT(input1)
    input2_inv = cmos_NOT(input2)

    pmos_A = pmos(input1, power)
    pmos_B = pmos(input2, pmos_A)

    pmos_A2 = pmos(input1_inv, power)
    pmos_B2 = pmos(input2_inv, pmos_A2)

    nmos_A = nmos(input1, power)
    nmos_B2 = nmos(input2_inv, nmos_A)

    nmos_A2 = nmos(input1_inv, power)
    nmos_B = nmos(input2, nmos_A2)

    if pmos_B is not None or pmos_B2 is not None:
        return power
    elif nmos_B is not None or nmos_B2 is not None:
        return GROUND
    else:
        return None  # Floating state (should not occur in valid CMOS design)

def cmos_XOR(input1: TRANSISTOR_OUTPUT, input2: TRANSISTOR_OUTPUT, ground: bool = GROUND) -> TRANSISTOR_OUTPUT:
    '''
    CMOS XOR constructed from fundamental CMOS building blocks:
    XOR = (A OR B) AND (A NAND B)
    
               Input1    Input2
                 │         │
        ┌────────┼─────────┼────────┐
        │        │         │        │
        ▼        ▼         ▼        ▼
     ┌──────────────┐   ┌──────────────┐
     │   cmos_OR    │   │  cmos_NAND   │
     └──────┬───────┘   └──────┬───────┘
            │                  │
         OR Out             NAND Out
            │                  │
            └────────┬─────────┘
                     │
                     ▼
             ┌──────────────┐
             │   cmos_AND   │
             └──────┬───────┘
                    │
                    ▼
                 OUTPUT
    '''
    ## - If (0, 0) --> 0
    ##    - CMOS OR returns 0, CMOS NAND returns 1
    ##    - CMOS AND returns 0
    ## - If (0, 1) --> 1
    ##    - CMOS OR returns 1, CMOS NAND returns 1
    ##    - CMOS AND returns 1    
    ## - If (1, 0) --> 1
    ##    - CMOS OR returns 1, CMOS NAND returns 1
    ##    - CMOS AND returns 1
    ## - If (1, 1) --> 0
    ##    - CMOS OR returns 1, CMOS NAND returns 0
    ##    - CMOS AND returns 0
    or_out = cmos_OR(input1, input2, ground=ground)
    nand_out = cmos_NAND(input1, input2, ground=ground)
    return cmos_AND(or_out, nand_out, ground=ground)